# CampusVoice-RT: 재현 노트북 (Colab, NVIDIA L4)

런타임 → GPU 유형을 **L4**로 설정한 뒤 위에서부터 실행. 모든 베이스라인(B1~B4)을 같은 런타임·같은 VAD 설정으로 측정한다.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/ekgP525/asr-challenge.git
%cd asr-challenge
!pip install -q -U "qwen-asr[vllm]" silero-vad soundfile librosa matplotlib
!python -c "from importlib.metadata import version as v; print('torch', v('torch'), 'vllm', v('vllm'), 'qwen-asr', v('qwen-asr'))"

## Hugging Face 인증

Colab 공용 IP는 익명 요청이 속도 제한(429)에 걸린다. 읽기 전용 토큰을 Colab 보안 비밀(🔑)에 `HF_TOKEN`으로 넣고 노트북 접근을 허용한 뒤 실행.

In [ ]:
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!huggingface-cli whoami

## 평가셋 다운로드 (Hugging Face Datasets, 링크 TBD)

In [ ]:
!pip install -q huggingface_hub
# audio.tar.gz = one request (2,125 single files trip the Hub rate limit: 1,000 API calls / 5 min)
!huggingface-cli download ekgP115/campusvoice-rt-eval --repo-type dataset --include manifest.jsonl audio.tar.gz --local-dir data/eval --quiet
!tar -xzf data/eval/audio.tar.gz -C data/eval && rm data/eval/audio.tar.gz
!for c in clean snr20 snr10 snr5 snr0; do echo -n "$c: "; ls data/eval/audio/$c | wc -l; done; wc -l data/eval/manifest.jsonl

## 베이스라인 실행 → 채점 (백그라운드, Drive 저장)

`scripts/colab_run_all.sh`를 nohup으로 띄운다. 브라우저를 닫아도 VM에서 계속 돌고, 10분마다 진행분을 Drive(`campusvoice-rt/`, 50 MB 미만)에 복사한다. 세션이 죽으면 설치·인증·다운로드 셀을 다시 거친 뒤 이 셀을 다시 실행하면 끝난 시스템은 건너뛰고 끊긴 시스템은 이어서 돈다. L4 기준 B1·B3 각 10~20분, B2 약 65분, B4 약 40분, 흔들림 검증 10분.

In [ ]:
from google.colab import drive; drive.mount('/content/drive')
import os
os.environ['SYSTEMS'] = 'B1 B2 B3 B4'   # 이미 Drive에 결과가 있는 시스템은 자동으로 건너뜀
!nohup bash scripts/colab_run_all.sh > overnight.log 2>&1 &
!sleep 5; tail -3 overnight.log

## 진행 확인

`ALL_DONE`이 보이면 끝. 흔들림 검증(B2 40발화 × 3회)도 스크립트가 마지막에 수행한다.

In [ ]:
!tail -5 overnight.log
!wc -l runs/*/predictions.jsonl 2>/dev/null
!ls /content/drive/MyDrive/campusvoice-rt/results 2>/dev/null

## 리더보드

In [ ]:
!python evaluation/make_leaderboard.py
from IPython.display import Image, Markdown, display
display(Markdown(open('results/leaderboard.md').read()))
display(Image('results/latency_cer_paths.png'))